In [5]:
import numpy as np 
import pandas as pd

In [6]:
df = pd.read_csv('diabetes.csv')

In [7]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [8]:
df.corr()['Outcome'].sort_values(ascending=True)

DiabetesPedigreeFunction    0.173844
BloodPressure               0.174469
Pregnancies                 0.221898
Age                         0.238356
SkinThickness               0.295138
BMI                         0.315577
Insulin                     0.377081
Glucose                     0.495990
Outcome                     1.000000
Name: Outcome, dtype: float64

In [9]:
X = df.iloc[:,:-1].values
y= df.iloc[:,-1].values

In [10]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler= StandardScaler()

In [12]:
X = scaler.fit_transform(X)

In [13]:
X.shape

(768, 8)

In [14]:
from sklearn.model_selection import  train_test_split

X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=41)

In [83]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout


In [19]:
model = Sequential()
model.add(Dense(32,activation='relu', input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

In [20]:
model.fit(X_train,y_train,batch_size=32,epochs=10, validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.6645 - loss: 0.6266 - val_accuracy: 0.6883 - val_loss: 0.5721
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7117 - loss: 0.5664 - val_accuracy: 0.7273 - val_loss: 0.5305
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7492 - loss: 0.5275 - val_accuracy: 0.7792 - val_loss: 0.5015
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7671 - loss: 0.5000 - val_accuracy: 0.7792 - val_loss: 0.4802
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.7801 - loss: 0.4802 - val_accuracy: 0.7922 - val_loss: 0.4652
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7964 - loss: 0.4659 - val_accuracy: 0.7987 - val_loss: 0.4543
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8013 - loss: 0.4541 - val_accuracy: 0.7857 - val_loss: 0.4453
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7997 - loss: 0.4450 - val_accuracy: 0.7922 - v

How to automate using Keras tuner to find the best suitable parameters for our model

1. How to select appropriate optimizer
2. How to select number of nodes in a layer
3. How to select number of hidden layers
4. Will try everything in one model


In [25]:
#!pip install keras-tuner
import keras_tuner as kt

In [27]:
#creating a build function

def build_model(hp):

    model = Sequential()

    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    optimizer =hp.Choice('optimizer',values= ['adam','sgd','rmsprop','Adadelta', 'Adagrad'])

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [36]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy', max_trials=3,overwrite=True  # <-- restarts clean if previous run failed
)

In [37]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))


Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.7402597665786743

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 13s


In [38]:
tuner.results_summary()

Results summary
Results in .\untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: adam
Score: 0.7792207598686218

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7402597665786743

Trial 1 summary
Hyperparameters:
optimizer: Adagrad
Score: 0.41558441519737244


In [42]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [43]:
model = tuner.get_best_models(num_models=1)[0]

c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


In [44]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

fitting the best model

In [45]:
model.fit(X_train,y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.7850 - loss: 0.4949 - val_accuracy: 0.7727 - val_loss: 0.4873
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7899 - loss: 0.4765 - val_accuracy: 0.7792 - val_loss: 0.4702
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7997 - loss: 0.4644 - val_accuracy: 0.7792 - val_loss: 0.4586
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8013 - loss: 0.4554 - val_accuracy: 0.7922 - val_loss: 0.4500
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.8029 - loss: 0.4470 - val_accuracy: 0.7987 - val_loss: 0.4416
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8078 - loss: 0.4395 - val_accuracy: 0.7922 - val_loss: 0.4346
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8078 - loss: 0.4332 - val_accuracy: 0.7792 - val_loss: 0.4287
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8046 - loss: 0.4276 - val_accurac

See accuracy has gone to 88% upon using model with best optimizer

Now, we will find the right number of neurons

In [54]:
def build_model(hp):
    
    model=Sequential()

    units = hp.Int('units',min_value=8,max_value=128, step=8)

    model.add(Dense(units=units, activation='relu', input_dim=8))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

Now we will make tuner object

In [55]:
tuner= kt.RandomSearch(build_model,objective='val_accuracy', max_trials=5,directory='my_dir')

In [56]:
tuner.search(X_train,y_train, epochs=5, validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 09s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 46s


In [57]:
tuner.results_summary()

Results summary
Results in my_dir\untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
units: 80
Score: 0.8116883039474487

Trial 3 summary
Hyperparameters:
units: 128
Score: 0.8051947951316833

Trial 0 summary
Hyperparameters:
units: 72
Score: 0.7857142686843872

Trial 4 summary
Hyperparameters:
units: 40
Score: 0.7792207598686218

Trial 2 summary
Hyperparameters:
units: 32
Score: 0.7467532753944397


In [59]:
tuner.get_best_hyperparameters()[0].values

{'units': 80}

In [62]:
model= tuner.get_best_models(num_models=1)[0]

c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


Training the best model

In [64]:
model.fit(X_train,y_train, batch_size=32, epochs=100, initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7980 - loss: 0.4363
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7980 - loss: 0.4264
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7980 - loss: 0.4193
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4139
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4089
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8046 - loss: 0.4046
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8078 - loss: 0.3995
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8111 - loss: 0.3956
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8127 - loss: 0.3908
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8176 - loss: 0.3865
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8192 - loss: 0.3832
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc

# Layers

Now we will check how to select numbers of Layers

In [65]:
def build_model(hp):

    model= Sequential()
    model.add(Dense(80,activation='relu', input_dim=8))

    for i in range(hp.Int('num_layers',min_value=1, max_value=10)):

        model.add(Dense(72, activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
    return model




In [66]:
tuner= kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='my_dir', project_name='sk_num_layers')

c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [70]:
tuner.search(X_train,y_train, epochs=5, validation_data=(X_test,y_test))

Trial 3 Complete [00h 01m 02s]
val_accuracy: 0.8701298832893372

Best val_accuracy So Far: 0.8701298832893372
Total elapsed time: 00h 01m 04s


In [71]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7}

In [72]:
model=tuner.get_best_models(num_models=1)[0]

c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(store)


In [73]:
model.fit(X_train,y_train, epochs=100, initial_epoch=5, validation_data=(X_test,y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.8648 - loss: 0.3365 - val_accuracy: 0.8571 - val_loss: 0.3684
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8990 - loss: 0.2857 - val_accuracy: 0.8571 - val_loss: 0.3842
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8941 - loss: 0.2838 - val_accuracy: 0.8377 - val_loss: 0.3624
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9039 - loss: 0.2843 - val_accuracy: 0.8442 - val_loss: 0.3853
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9072 - loss: 0.2588 - val_accuracy: 0.8571 - val_loss: 0.3514
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9007 - loss: 0.2591 - val_accuracy: 0.8701 - val_loss: 0.3386
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9218 - loss: 0.2283 - val_accuracy: 0.8247 - val_loss: 0.4943
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9251 - loss: 0.2154 - val_accuracy

checking mutiple things now


In [84]:
def build_model(hp):

    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        if counter == 0:
            # First layer (requires input_dim)
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                    activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
                    input_dim=8
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])))
        else:
            # Subsequent hidden layers
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                    activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])))
        counter += 1

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile model with tuned optimizer
    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model


In [86]:
tuner= kt.RandomSearch(build_model,objective='val_accuracy', max_trials=3,  directory='my_dir', project_name='finding_all_comb_1')

c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [87]:
tuner.search(X_train,y_train, epochs=5, validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 18s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 50s


In [88]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10,
 'units0': 96,
 'activation0': 'tanh',
 'dropout0': 0.2,
 'optimizer': 'adam',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1,
 'units2': 8,
 'activation2': 'relu',
 'dropout2': 0.1,
 'units3': 8,
 'activation3': 'relu',
 'dropout3': 0.1,
 'units4': 8,
 'activation4': 'relu',
 'dropout4': 0.1,
 'units5': 8,
 'activation5': 'relu',
 'dropout5': 0.1,
 'units6': 8,
 'activation6': 'relu',
 'dropout6': 0.1,
 'units7': 8,
 'activation7': 'relu',
 'dropout7': 0.1,
 'units8': 8,
 'activation8': 'relu',
 'dropout8': 0.1,
 'units9': 8,
 'activation9': 'relu',
 'dropout9': 0.1}

In [80]:
model=tuner.get_best_models(num_models=1)[0]

c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\sujat\projects\DeepLearning\100-Days-of-Deep-Learning\myvenv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(store)


In [82]:
model.fit(X_train,y_train, epochs=200, initial_epoch=5, validation_data=(X_test,y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8192 - loss: 0.3926 - val_accuracy: 0.8117 - val_loss: 0.4150
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8241 - loss: 0.3720 - val_accuracy: 0.8377 - val_loss: 0.4011
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8420 - loss: 0.3536 - val_accuracy: 0.8377 - val_loss: 0.3912
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8599 - loss: 0.3444 - val_accuracy: 0.8377 - val_loss: 0.3863
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8583 - loss: 0.3354 - val_accuracy: 0.8442 - val_loss: 0.3761
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8648 - loss: 0.3288 - val_accuracy: 0.8506 - val_loss: 0.3709
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8599 - loss: 0.3288 - val_accuracy: 0.8442 - val_loss: 0.3684
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8664 - loss: 0.3217 - val_accuracy